In [1]:
import pandas as pd

# Run GeneID

In [3]:
#geneid_command="time geneid -3P {parameter_file} {assembly} > {annotation.gff3}"

tr_sp=!ls ../data/species

for src in ["own", "git"]:
    #generate 1 file per source
    with open(f"../job_commands/SARpred_{src}.txt", "w") as out:
        #create predictions folder command
        pred_folder="../results/pred"
        print(f"mkdir -p {pred_folder}", file=out)
        for sp in tr_sp:
            
            #reference and model name are adaptable to the source
            ref=!realpath ../data/species/$sp/CLEAN*fna
            model=f"{pred_folder}/{sp}_{src}.gff3"

            #parameter location changes depending on source
            param=!realpath ../results/trainedParams/$sp*.param
            param=param[0]
            #use github parameters
            if src=="git":
                filename_git=f"../data/geneid-parameter-files/parameter_files/{sp}*.param"

                try: #if no github parameters exist, dont runn geneID for git parameters

                    found_files=!realpath $filename_git 2>/dev/null #supress error output
                    #set up error for species with no github parameters
                    if "*" in found_files[0]: #found files is exactly filename_git if nothing is found
                        raise FileNotFoundError(f"There is no github parameters for this species")
                    
                except Exception as e:
                    print(f"{sp} presents error: {e}")
                    continue

                param=found_files[0]

            #geneid param_own/param_git fasta_assembly > gff_model
            command=f"time geneid -3P {param} {ref[0]} > {model}"

            #link inside the specie folder just in case
            logsDir_cmd=f"../results/specie_logs/{sp}/" #if running from exsting parameters create logs folder
            lnPred_cmd=f"ln -vf {model} ../results/specie_logs/{sp}/"

            print(command)
            print(command, file=out)
            print(logsDir_cmd, file=out)
            print(lnPred_cmd, file=out)
            

#Execute commands to predict CDS

time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/trainedParams/Cloeon_dipterum.geneid.optimized.param /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Cloeon_dipterum/CLEAN_Cdipterum_gn.fna > ../results/pred/Cloeon_dipterum_own.gff3


time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/trainedParams/Daphnia_magna.geneid.optimized.param /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Daphnia_magna/CLEAN_Dmagna_gn.fna > ../results/pred/Daphnia_magna_own.gff3
time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/trainedParams/Drosophila_melanogaster.geneid.optimized.param /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Drosophila_melanogaster/CLEAN_Dmelanogaster_gn.fna > ../results/pred/Drosophila_melanogaster_own.gff3
time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/trainedParams/Drosophila_willistoni.geneid.optimized.param /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Drosophila_willistoni/CLEAN_Dwillistoni_gn.fna > ../results/pred/Drosophila_willistoni_own.gff3
time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/gene

# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

In [4]:
from CDS_machine import CDS_2_gffcomp

tr_sp=!ls ../data/species

for src in ["own", "git"]:
    with open(f"../job_commands/gffcomp_{src}.txt", "w") as f:
        
        #create folder to store comparison information
        resloc=f"../results/gffcmp"
        print(f"mkdir -p {resloc}", file=f)
        for sp in tr_sp:

            try:
                #if no github parameters exist, dont process this species git comparison
                filename_git=f"../data/geneid-parameter-files/parameter_files/{sp}*.param"
                found_files=!realpath $filename_git 2>/dev/null #supress error output

                if not found_files:
                    raise FileNotFoundError(f"There is no github parameters for this species")
                
                #reference annotation
                RefAnn=!realpath ../data/species/$sp/CDS_*gff
                RefAnn=RefAnn[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/pred/$sp*_$src*.gff3
                pred=pred[0]
                
                #adapt cds to be able to be compared
                adapted_RefAnn=f"../results/specie_logs/{sp}/CDSgff_{sp}.gff"
                
                CDS_2_gffcomp(RefAnn, adapted_RefAnn)

                #comparison command
                comparing_cmd=f"gffcompare -r {adapted_RefAnn} {pred} -o {resloc}/{sp}-cmp"
                
                #move all except stats, which is linked
                moveComparisons_cmd=f"mv {resloc}/!(*.stats*) ../results/specie_logs/{sp}/"
                lnStat_cmd=f"ln -vf {resloc}/{sp}-cmp.stats ../results/specie_logs/{sp}/"
                
                print(comparing_cmd, file=f)
                print(comparing_cmd)
                print(moveComparisons_cmd, file=f)
                print(lnStat_cmd, file=f)
                print(f'echo "done_{sp}"', file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue

gffcompare -r ../results/specie_logs/Cloeon_dipterum/CDSgff_Cloeon_dipterum.gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Cloeon_dipterum_own.gff3 -o ../results/gffcmp/Cloeon_dipterum-cmp
gffcompare -r ../results/specie_logs/Daphnia_magna/CDSgff_Daphnia_magna.gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Daphnia_magna_own.gff3 -o ../results/gffcmp/Daphnia_magna-cmp
gffcompare -r ../results/specie_logs/Drosophila_melanogaster/CDSgff_Drosophila_melanogaster.gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Drosophila_melanogaster_own.gff3 -o ../results/gffcmp/Drosophila_melanogaster-cmp
gffcompare -r ../results/specie_logs/Drosophila_willistoni/CDSgff_Drosophila_willistoni.gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Drosophila_willistoni_own.gff3 -o ../results/gffcmp/Drosophila_willistoni-cmp
gffcompare -r ../results/specie_logs/Mnemiopsis_leidyi/CDSgff_Mn

In [6]:
from statsCompression import gffstats_2data

tr_sp=!ls ../data/species

statsData=[]
for sp in tr_sp:
    print(f"Species is {sp}")
    statsfile=f"../results/gffcmp/{sp}-cmp.stats"

    #return data for a concrete stats file
    parsed_dict=gffstats_2data(statsfile)
    #joind data from all stats files
    statsData.append(parsed_dict)

#transform data list to dataframe
df=pd.DataFrame(statsData)
df=df.fillna("NA")

df.to_csv("../results/gffcmp/specie_stats.csv", index=False, sep="\t")
print("Done")


Species is Cloeon_dipterum
['28550/139208', '(', '20.5%)']
['22303/120627', '(', '18.5%)']
['12154/115178', '(', '10.6%)']
['12094/101835', '(', '11.9%)']
['791/16339', '(', '4.8%)']
['3632/18792', '(', '19.3%)']
Species is Daphnia_magna
['51521/157852', '(', '32.6%)']
['53309/156545', '(', '34.1%)']
['31118/125518', '(', '24.8%)']
['32689/119852', '(', '27.3%)']
['7120/31343', '(', '22.7%)']
['12905/36693', '(', '35.2%)']
Species is Drosophila_melanogaster
['11535/62990', '(', '18.3%)']
['11415/58451', '(', '19.5%)']
['4922/47893', '(', '10.3%)']
['6069/41928', '(', '14.5%)']
['1232/13929', '(', '8.8%)']
['4201/16523', '(', '25.4%)']
Species is Drosophila_willistoni
['7970/61747', '(', '12.9%)']
['46017/97294', '(', '47.3%)']
['3760/46996', '(', '8.0%)']
['25649/65580', '(', '39.1%)']
['622/14778', '(', '4.2%)']
['17375/31714', '(', '54.8%)']
Species is Mnemiopsis_leidyi
['21662/127104', '(', '17.0%)']
['43418/147482', '(', '29.4%)']
['8516/107307', '(', '7.9%)']
['22009/119929', '(',

### Result comparison

#### .stats files present info

**According to previous statistics**

- git and own have pretty similar results in both species, but in Pvivax they are closer(more similar) than in Pfalc. We can see this in the probability spreads, both are more disperse in Pfalc
- Individual distributuins show slight skew toward own data while transition matrix probabilities have a more stable distribution


**Both are confirmed by looking at the .stats files of the geneID results:**

- Git and Own results are pretty similar, specially in Pvivax
- Own results are slightly better


**Summary:** own produces better results

## BUSCO assessment

In [ ]:
#agat for gff>prot
tr_sp=!ls ../data/species
tr_sp=["Drosophila_willistoni"]

#for src in ["git", "own"]:
for src in ["own"]:
    with open(f"../job_commands/buscoComp_{src}.txt", "w") as f:
        for sp in tr_sp:
            try:
                #if no github parameters exist, dont process this species git comparison
                filename_git=f"../data/geneid-parameter-files/parameter_files/{sp}*.param"
                found_files=!realpath $filename_git 2>/dev/null #supress error output

                if not found_files:
                    #@remove created file?? #@correct path errors
                    raise FileNotFoundError(f"There is no github parameters for this species")
                
                #reference sequence
                RefSeq=!realpath ../data/species/$sp/raw_*gn*
                RefSeq=RefSeq[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/pred/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store outputs
                prot_res=f"../results/specie_logs/{sp}/{sp}_prot.fa" #protein sequence output
                busco_outPath=f"../results/busco/" #busco comparisons output
                busco_res=f"{sp}_busco" #busco out name

                folder_cmd=f"mkdir -p {resloc}"

                #gff to protein command
                TOprotein_cmd=f"agat_sp_extract_sequences.pl \
                                -f {RefSeq} \
                                -g {pred} \
                                -t CDS \
                                -p \
                                -o {prot_res}"
                TOprotein_cmd=f"gffread -g {RefSeq} {pred} -y {prot_res}"

                print(TOprotein_cmd)
                print(TOprotein_cmd, file=f)
                
                busco_cmd=f"busco -m protein \
                            -i {prot_res} \
                            --auto-lineage \
                            --cpu $(nproc) \
                            -f \
                            --out_path {busco_outPath} \
                            -o {busco_res}" #--offline #autolineage creates errors
                busco_cmd=busco_cmd.replace("                             ", " ")

                busco_plot=f"busco --plot {busco_outPath}{busco_res}"
                
                print(busco_cmd)
                print(busco_cmd, file=f)
                
                print(busco_plot)
                print(busco_plot, file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            

gffread -g /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Drosophila_melanogaster/raw_Dmelanogaster_gn.fna /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Drosophila_melanogaster_own.gff3 -y ../results/specie_logs/Drosophila_melanogaster/Drosophila_melanogaster_prot.fa
busco -m protein -i ../results/specie_logs/Drosophila_melanogaster/Drosophila_melanogaster_prot.fa --auto-lineage --cpu $(nproc) -f --out_path ../results/busco/ -o Drosophila_melanogaster_busco
busco --plot ../results/busco/Drosophila_melanogaster_busco
gffread -g /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Drosophila_willistoni/raw_Dwillistoni_gn.fna /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/pred/Drosophila_willistoni_own.gff3 -y ../results/specie_logs/Drosophila_willistoni/Drosophila_willistoni_prot.fa
busco -m protein -i ../results/specie_logs/Drosophila_willistoni/Drosophila_willistoni_prot.fa --auto-

# Setup for IGV transcript comparison

In [ ]:
#generate symlinks to a igv allfiles folder
#s_results/specie/gff_*.gtf
#s_results/specie/gff_*.stats
#s_results/specie/*.gff3

    
#symlink files
#for raw files
clean_seq=!realpath ../data/species/**/CLEAN*.fna
clean_ann=!realpath ../data/species/**/CDS*ann.gff
rawData=clean_seq+clean_ann

for file in rawData:
    sp=file.split("/")[-2]
    #remove species shortened name from filename
    purename=file.split("/")[-1].split("_")
    purename="_".join([purename[0], purename[-1]])
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vf $file $dest/og_$purename


#for prediction result files
gff3=!realpath ../results/**/*.gff3

for file in gff3:
    sp=file.split("/")[-2]
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vf $file $dest


#for gffcompare files
gtfComp=!realpath ../results/**/gffcmp/*.gtf
statsComp=!realpath ../results/**/gffcmp/*.stats
gffComp=gtfComp+statsComp

for file in gffComp:
    sp=file.split("/")[-3]
    #name cleanup
    purename=file.split("/")[-1]
    purename=purename.replace(".annotated", "_ann").replace("GFF", "prediction")
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vf $file $dest/$purename






'../nres/Plasmodium_falciparum/og_CLEAN_gn.fna' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_falciparum/CLEAN_Pfalci_gn.fna'
'../nres/Plasmodium_vivax/og_CLEAN_gn.fna' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_vivax/CLEAN_Pvivax_gn.fna'
'../nres/Plasmodium_falciparum/og_CDS_ann.gff' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_falciparum/CDS_Pfalci_ann.gff'
'../nres/Plasmodium_vivax/og_CDS_ann.gff' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_vivax/CDS_Pvivax_ann.gff'
'../nres/Plasmodium_falciparum/Plasmodium_falciparum_git.gff3' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/results/Plasmodium_falciparum/Plasmodium_falciparum_git.gff3'
'../nres/Plasmodium_falciparum/Plasmodium_falciparum_own.gff3' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/results/Plasmodium_falciparum/Plasmodium_falciparum_own.gff3'
'../nres/Plasmodium_vivax/Plasmodium_vivax_git